# v16 — Focused investigation: Technical, Valuation and Micro
Supporting research for the final deliverables (`Markets_Scorecard_v16_FINAL.ipynb`). v1–v15 are untouched.
**Data policy:** only the workbooks supplied with the assignment. No external data.

| Section | Question |
|---|---|
| 1–2 | Is the reconstructed price series valid, and what is it made of? |
| 3 | **Technical (priority):** does any price-path signal survive, once overlap with Micro and Valuation is removed? |
| 4 | Reconcile the technical examined IC +0.163 against the walk-forward IC +0.058 |
| 5 | **Micro:** do revision acceleration and revision momentum add to the 6M revision? |
| 6 | **Valuation:** can the negative out-of-sample result for the earnings-yield spread be resolved? |
| 7 | Frozen definitions for the final scorecard |

**Discipline:** hypothesis sets are declared before evaluation; signal *direction* is fixed on discovery evidence and frozen before the validation window is read; the 2017–2026 window is labelled EXAMINED, never untouched holdout.

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings, json, subprocess
from pathlib import Path
import statsmodels.api as sm
from scipy.stats import rankdata
from IPython.display import display
warnings.filterwarnings('ignore')
pd.options.display.float_format = '{:,.3f}'.format; pd.options.display.width = 250; pd.options.display.max_columns = 40; pd.options.display.max_rows = 300
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25, 'font.size': 10})
ROOT = Path('..').resolve(); OUT = ROOT/'output'/'v16'; CH = OUT/'charts'
for p in (OUT, CH): p.mkdir(parents=True, exist_ok=True)
COUNTRIES = ['US', 'Eurozone', 'UK', 'Switzerland', 'Japan']; SEED = 7; NBOOT = 2000
SPEC = {'US': dict(eps='Monthly_EPS', fci=('GSFCI Index', -1), y2='USGGT02Y Index', fx=None, file='US_raw_data.xlsx'),
        'Eurozone': dict(eps='BEst EPS', fci=('BFCIEU Index', +1), y2='GDBR2 Index', fx=('EURUSD BGN Curncy', 'direct'), file='EUR_raw_data.xlsx'),
        'UK': dict(eps='BEst EPS', fci=('GSGBFCI Index', -1), y2='GUKG2 Index', fx=('GBPUSD BGN Curncy', 'direct'), file='UK_raw_data.xlsx'),
        'Switzerland': dict(eps='BEst EPS', fci=('GSEAFCI Index', -1), y2='GSWISS02 Index', fx=('USDCHF Curncy', 'invert'), file='CH_raw_data.xlsx'),
        'Japan': dict(eps='BEst EPS', fci=('GSJPFCI Index', -1), y2='GJGB2 Index', fx=('USDJPY BGN Curncy', 'invert'), file='JPY_raw_data.xlsx')}
def monthly(df):
    d = df.dropna(subset=['Date']).copy(); d['month'] = pd.to_datetime(d['Date']).dt.to_period('M'); d['_d'] = pd.to_datetime(d['Date'])
    return d.sort_values('_d').drop_duplicates('month', keep='last').set_index('month')
RAW = {c: monthly(pd.read_excel(ROOT/'data'/'raw'/SPEC[c]['file'])) for c in COUNTRIES}
def grab(c, k):
    s = SPEC[c]; d = RAW[c]
    if k == 'fci': return s['fci'][1] * d[s['fci'][0]]
    if k == 'fx': return pd.Series(1.0, index=d.index) if s['fx'] is None else (d[s['fx'][0]] if s['fx'][1] == 'direct' else 1.0 / d[s['fx'][0]])
    return d[{'pe': 'BEst P/E Ratio', 'eps': s['eps'], 'y2': s['y2']}[k]]
IDX = pd.period_range(max(grab(c, 'pe').first_valid_index() for c in COUNTRIES), min(RAW[c].index.max() for c in COUNTRIES), freq='M')
PE, EPS, FCI, Y2, FX = [pd.DataFrame({c: grab(c, k) for c in COUNTRIES}).reindex(IDX) for k in ('pe', 'eps', 'fci', 'y2', 'fx')]
PX = PE * EPS; PX_USD = PX * FX; AS_OF = IDX[-1]
EMB_END, EXAM_START = pd.Period('2016-12', 'M'), pd.Period('2017-07', 'M')
def rel(df, h):
    r = df.shift(-h) / df - 1; return r.sub(r.mean(axis=1), axis=0)
TGT = {('ret', h): rel(PX_USD, h) for h in (3, 6)} | {('mult', h): rel(PE, h) for h in (3, 6)} | {('earn', h): rel(EPS, h) for h in (3, 6)}
cs_z = lambda d: d.sub(d.mean(axis=1), axis=0).div(d.std(axis=1, ddof=1), axis=0).where(d.notna().all(axis=1))
pit_z = lambda d, mp=36: (d - d.expanding(mp).mean()) / d.expanding(mp).std()
own_sd = lambda d, mp=36: d / d.expanding(mp).std()
def unit_ranks(A):
    r = rankdata(A, axis=-1) - (A.shape[-1] + 1) / 2.0; return r / np.sqrt((r ** 2).sum(-1, keepdims=True))
def ic_parts(S, Y):
    s = S.to_numpy(float); y = Y.reindex(S.index).to_numpy(float); ok = ~np.isnan(s).any(1) & ~np.isnan(y).any(1)
    return pd.DataFrame(unit_ranks(s[ok]) * unit_ranks(y[ok]), index=S.index[ok], columns=S.columns)
def ic_series(S, Y): return ic_parts(S, Y).sum(axis=1)
def hac_t(x, lags):
    x = np.asarray(x, float); return float(sm.OLS(x, np.ones(len(x))).fit(cov_type='HAC', cov_kwds={'maxlags': lags}).tvalues[0]) if len(x) > 10 else np.nan
def mbb(x, L, n=NBOOT, seed=SEED):
    x = np.asarray(x, float); T = len(x)
    if T < 3 * L: return (np.nan, np.nan)
    rng = np.random.default_rng(seed); nb = int(np.ceil(T / L))
    st = rng.integers(0, T - L + 1, (n, nb)); m = x[(st[:, :, None] + np.arange(L)).reshape(n, -1)[:, :T]].mean(1)
    return tuple(np.percentile(m, [2.5, 97.5]))
def summarise(S, h, tgt='ret'):
    ic = ic_series(S, TGT[(tgt, h)]); d = ic[ic.index <= EMB_END]; e = ic[ic.index >= EXAM_START]; lo, hi = mbb(ic.values, 2 * h)
    return {'n': len(ic), 'IC disc': d.mean() if len(d) else np.nan, 'IC exam': e.mean() if len(e) else np.nan, 'IC full': ic.mean(),
            'HAC t': hac_t(ic.values, h), 'CI lo': lo, 'CI hi': hi}
print(f'Panel {IDX[0]} -> {AS_OF} ({len(IDX)} months).')

Panel 2005-12 -> 2026-04 (245 months).


## 1. Reconstructed price series — validation and coverage
There is no index price in the dataset (v15 §1c), so the price series is reconstructed as $P=(P/E)\times EPS$. Three checks before any technical work.

In [2]:
RLOC = PX.pct_change()
COVER = pd.DataFrame({'first month': PX.apply(lambda s: str(s.first_valid_index())), 'last month': PX.apply(lambda s: str(s.last_valid_index())),
                      'months': PX.notna().sum(), 'internal gaps': PX.apply(lambda s: int(s.loc[s.first_valid_index():s.last_valid_index()].isna().sum())),
                      'ann. volatility %': RLOC.std() * np.sqrt(12) * 100, 'worst 12M drawdown %': PX.apply(lambda s: (s / s.cummax() - 1).min() * 100),
                      'monthly skew': RLOC.skew(), 'months with |return| > 20%': (RLOC.abs() > .20).sum()})
display(COVER.round(2))
known = {'2008-09': -9.08, '2008-10': -16.94, '2009-02': -10.99, '2011-09': -7.18, '2018-12': -9.18, '2020-03': -12.51, '2020-04': 12.68, '2022-04': -8.80}
SPX = pd.DataFrame([{'month': k, 'reconstructed US %': RLOC['US'].loc[pd.Period(k, 'M')] * 100, 'published S&P 500 %': v} for k, v in known.items()]).set_index('month')
SPX['difference (pp)'] = SPX['reconstructed US %'] - SPX['published S&P 500 %']; display(SPX.round(2))
print(f'Mean absolute difference vs the published S&P 500: {SPX["difference (pp)"].abs().mean():.3f} pp across {len(SPX)} months.')
print('The US reconstruction is validated. The other four CANNOT be validated against a published index with the supplied data —')
print('their plausibility rests on the identity P/E = Price/EPS holding for the same index and vintage, and on realistic volatility and drawdowns above.')
CORR = RLOC.corr(); display(CORR.round(2))
print('Cross-market monthly return correlations (real developed-market equities typically run 0.6-0.85).')

,first month,last month,months,internal gaps,ann. volatility %,worst 12M drawdown %,monthly skew,months with |return| > 20%
US,2005-12,2026-04,245,0,15.190,-52.570,-0.580,0
Eurozone,2005-12,2026-04,245,0,16.460,-57.990,-0.440,0
UK,2005-12,2026-04,245,0,13.070,-43.580,-0.530,0
Switzerland,2005-12,2026-04,245,0,12.480,-50.840,-0.400,0
Japan,2005-12,2026-04,245,0,17.230,-60.700,-0.500,1


,reconstructed US %,published S&P 500 %,difference (pp)
month,,,
2008-09,-9.090,-9.080,-0.010
2008-10,-16.970,-16.940,-0.030
2009-02,-11.000,-10.990,-0.010
2011-09,-7.170,-7.180,0.010
2018-12,-9.350,-9.180,-0.170
2020-03,-12.530,-12.510,-0.020
2020-04,12.700,12.680,0.020
2022-04,-8.830,-8.800,-0.030


Mean absolute difference vs the published S&P 500: 0.039 pp across 8 months.
The US reconstruction is validated. The other four CANNOT be validated against a published index with the supplied data —
their plausibility rests on the identity P/E = Price/EPS holding for the same index and vintage, and on realistic volatility and drawdowns above.


,US,Eurozone,UK,Switzerland,Japan
US,1.000,0.830,0.730,0.740,0.660
Eurozone,0.830,1.000,0.840,0.800,0.700
UK,0.730,0.840,1.000,0.710,0.610
Switzerland,0.740,0.800,0.710,1.000,0.620
Japan,0.660,0.700,0.610,0.620,1.000


Cross-market monthly return correlations (real developed-market equities typically run 0.6-0.85).


## 2. What the reconstructed price actually moves on
$\log(P_t/P_{t-k}) = \log(PE_t/PE_{t-k}) + \log(EPS_t/EPS_{t-k})$ — exact. Every technical signal is therefore a function of the same two inputs that drive the Valuation and Micro pillars.

In [3]:
rows = []
for k in (1, 3, 6, 12):
    dm = np.log(PE / PE.shift(k)); de = np.log(EPS / EPS.shift(k)); tot = np.log(PX / PX.shift(k))
    j = pd.concat([dm.stack().rename('m'), de.stack().rename('e'), tot.stack().rename('t')], axis=1).dropna()
    rows.append({'horizon': f'{k}M', 'identity error (max abs)': float((tot - (dm + de)).abs().max().max()),
                 'sd P/E leg': j.m.std(), 'sd EPS leg': j.e.std(), 'sd total': j.t.std(), 'corr(P/E leg, EPS leg)': j.m.corr(j.e),
                 'variance share P/E leg': j.m.cov(j.t) / j.t.var(), 'variance share EPS leg': j.e.cov(j.t) / j.t.var()})
DECOMP = pd.DataFrame(rows).set_index('horizon'); display(DECOMP.round(3))
DECOMP.to_csv(OUT/'price_decomposition.csv')
print('The P/E leg dominates at 1M and 3M (87%, 66% of variance); the EPS leg dominates at 6M and 12M (52%, 67%). The two legs offset each other at every horizon (corr -0.48 to -0.61).')
print('CONSEQUENCE: price momentum is mostly a VALUATION-change signal with an earnings-change signal subtracted. It is not an independent fourth source of information.')

,identity error (max abs),sd P/E leg,sd EPS leg,sd total,"corr(P/E leg, EPS leg)",variance share P/E leg,variance share EPS leg
horizon,,,,,,,
1M,0.000,0.049,0.031,0.044,-0.484,0.874,0.126
3M,0.000,0.090,0.079,0.078,-0.588,0.660,0.340
6M,0.000,0.131,0.134,0.117,-0.611,0.477,0.523
12M,0.000,0.171,0.198,0.168,-0.590,0.327,0.673


The P/E leg dominates at 1M and 3M (87%, 66% of variance); the EPS leg dominates at 6M and 12M (52%, 67%). The two legs offset each other at every horizon (corr -0.48 to -0.61).
CONSEQUENCE: price momentum is mostly a VALUATION-change signal with an earnings-change signal subtracted. It is not an independent fourth source of information.


## 3. Technical — priority investigation
### 3a. Declared candidate set and the direction protocol
Five price-path measures (the brief's list), each a function of the reconstructed price only.
**Direction protocol:** trend *continuation* (+) and *reversal* (−) are treated as two separate hypotheses for every measure.
The sign is chosen on **discovery evidence only**, then frozen. Validation is read afterwards and the sign is never flipped.
This is six binary selections; the multiplicity is disclosed and carried into the verdict.

In [4]:
def trend_q(P, w=12):
    lp = np.log(P); t = pd.Series(np.arange(len(lp)), index=lp.index, dtype=float)
    cr = lp.apply(lambda s: s.rolling(w).corr(t)); return (cr ** 2) * np.sign(cr)
TECH_RAW = {
 'A 6M momentum':              PX / PX.shift(6) - 1,
 'B 12−1M momentum':           PX.shift(1) / PX.shift(13) - 1,
 'C 1M return':                PX / PX.shift(1) - 1,
 'D 3M return':                PX / PX.shift(3) - 1,
 'E Signed trend quality 12M': trend_q(PX, 12),
 'F Vol-adjusted mom 12−1M':  (PX.shift(1) / PX.shift(13) - 1) / RLOC.rolling(12).std(),
}
TECH_HYP = {'A 6M momentum': 'continuation: trends persist over the forecast horizon',
            'B 12−1M momentum': 'continuation: classic momentum skipping the 1M reversal',
            'C 1M return': 'reversal expected: short-horizon liquidity/overreaction effects',
            'D 3M return': 'either: 3M sits between the reversal and momentum horizons',
            'E Signed trend quality 12M': 'continuation: smooth directional trends reflect gradual information diffusion',
            'F Vol-adjusted mom 12−1M': 'continuation: the same drift is more informative when the path is calm'}
rows = []
for k, raw in TECH_RAW.items():
    z = cs_z(raw.reindex(IDX))
    for sgn, lab in ((+1, 'continuation (+)'), (-1, 'reversal (−)')):
        ic = ic_series(sgn * z, TGT[('ret', 6)]); d = ic[ic.index <= EMB_END]
        rows.append({'measure': k, 'direction': lab, 'discovery IC 6M': d.mean(), 'discovery months': len(d)})
DIR = pd.DataFrame(rows).pivot(index='measure', columns='direction', values='discovery IC 6M')
DIR['FROZEN direction'] = np.where(DIR['continuation (+)'] >= DIR['reversal (−)'], 'continuation (+)', 'reversal (−)')
DIR['declared hypothesis'] = [TECH_HYP[k] for k in DIR.index]
display(DIR.round(3)); DIR.to_csv(OUT/'technical_direction_choice.csv')
SIGN = {k: (+1 if DIR.loc[k, 'FROZEN direction'] == 'continuation (+)' else -1) for k in DIR.index}
print('Signs are now FROZEN. Everything below uses them; no sign is revisited after reading validation.')
TZ = {k: SIGN[k] * cs_z(raw.reindex(IDX)) for k, raw in TECH_RAW.items()}

direction,continuation (+),reversal (−),FROZEN direction,declared hypothesis
measure,,,,
A 6M momentum,0.045,-0.045,continuation (+),continuation: trends persist over the forecast...
B 12−1M momentum,0.119,-0.119,continuation (+),continuation: classic momentum skipping the 1M...
C 1M return,-0.014,0.014,reversal (−),reversal expected: short-horizon liquidity/ove...
D 3M return,-0.024,0.024,reversal (−),either: 3M sits between the reversal and momen...
E Signed trend quality 12M,0.121,-0.121,continuation (+),continuation: smooth directional trends reflec...
F Vol-adjusted mom 12−1M,0.115,-0.115,continuation (+),continuation: the same drift is more informati...


Signs are now FROZEN. Everything below uses them; no sign is revisited after reading validation.


### 3b. How much of each technical candidate is just Micro and Valuation?
Each month the technical score is regressed cross-sectionally on the Micro score (EPS revision 6M) and the Valuation score (earnings-yield spread), and the **residual** is re-tested.
A candidate that loses its IC after this is reproducing the other pillars rather than adding a fourth source of information.

In [5]:
MICRO_Z = cs_z(EPS / EPS.shift(6) - 1)
VAL_Z = cs_z(pit_z(100.0 / PE - Y2))
def orth(S, ctrls):
    out = pd.DataFrame(index=S.index, columns=S.columns, dtype=float)
    for t in S.index:
        y = S.loc[t].to_numpy(float); X = np.column_stack([c.loc[t].to_numpy(float) for c in ctrls]) if ctrls else None
        if np.isnan(y).any() or X is None or np.isnan(X).any(): continue
        X = np.column_stack([np.ones(len(y)), X])
        out.loc[t] = y - X @ np.linalg.lstsq(X, y, rcond=None)[0]
    return cs_z(out)
rows = []
for k, S in TZ.items():
    r = {'candidate': k, 'frozen direction': DIR.loc[k, 'FROZEN direction']}
    for h in (3, 6):
        raw_s = summarise(S, h); res_s = summarise(orth(S, [MICRO_Z, VAL_Z]), h)
        r |= {f'{h}M raw IC full': raw_s['IC full'], f'{h}M residual IC full': res_s['IC full'],
              f'{h}M raw IC exam': raw_s['IC exam'], f'{h}M residual IC exam': res_s['IC exam'],
              f'{h}M IC retained': res_s['IC full'] / raw_s['IC full'] if abs(raw_s['IC full']) > 1e-9 else np.nan}
    pooled = pd.concat([S.stack().rename('t'), MICRO_Z.stack().rename('m'), VAL_Z.stack().rename('v')], axis=1).dropna()
    r['R² on Micro+Valuation'] = sm.OLS(pooled.t.values, sm.add_constant(pooled[['m', 'v']].values)).fit().rsquared
    rows.append(r)
ORTH = pd.DataFrame(rows).set_index('candidate')
cols = ['frozen direction', 'R² on Micro+Valuation', '6M raw IC full', '6M residual IC full', '6M IC retained', '6M raw IC exam', '6M residual IC exam']
display(ORTH[cols].style.format({c: '{:+.3f}' for c in cols if 'IC' in c and 'retained' not in c} | {'R² on Micro+Valuation': '{:.1%}', '6M IC retained': '{:.0%}'}))
ORTH.to_csv(OUT/'technical_orthogonalisation.csv')

,frozen direction,R² on Micro+Valuation,6M raw IC full,6M residual IC full,6M IC retained,6M raw IC exam,6M residual IC exam
candidate,,,,,,,
A 6M momentum,continuation (+),19.2%,+0.065,+0.021,32%,+0.103,+0.065
B 12−1M momentum,continuation (+),24.6%,+0.086,-0.010,-11%,+0.066,+0.003
C 1M return,reversal (−),2.0%,-0.014,-0.022,159%,-0.049,-0.028
D 3M return,reversal (−),7.2%,-0.013,-0.020,149%,-0.063,-0.037
E Signed trend quality 12M,continuation (+),21.6%,+0.133,+0.057,43%,+0.163,+0.039
F Vol-adjusted mom 12−1M,continuation (+),17.6%,+0.097,+0.032,33%,+0.101,+0.032


### 3c. Full battery on every candidate: 3M/6M, discovery/validation, ex-US, leave-one-out

In [6]:
def loo_ic(raw, sgn, drop, h):
    keep = [c for c in COUNTRIES if c != drop]
    S = sgn * cs_z(raw[keep].reindex(IDX))
    y = PX_USD[keep].shift(-h) / PX_USD[keep] - 1; y = y.sub(y.mean(axis=1), axis=0)
    return ic_series(S, y)
rows = []
for k, raw in TECH_RAW.items():
    for h in (3, 6):
        s = summarise(TZ[k], h)
        r = {'candidate': k, 'horizon': f'{h}M', **{kk: s[kk] for kk in ('n', 'IC disc', 'IC exam', 'IC full', 'HAC t', 'CI lo')}}
        loos = {c: loo_ic(raw, SIGN[k], c, h).mean() for c in COUNTRIES}
        r |= {f'ex-{c}': v for c, v in loos.items()}; r['min LOO'] = min(loos.values())
        rows.append(r)
TBAT = pd.DataFrame(rows).set_index(['candidate', 'horizon'])
display(TBAT.style.format({c: '{:+.3f}' for c in TBAT.columns if c != 'n' and c != 'HAC t'} | {'HAC t': '{:+.2f}', 'n': '{:.0f}'}))
TBAT.to_csv(OUT/'technical_battery.csv')

### 3d. Walk-forward: which technical candidate would have been chosen in real time?

In [7]:
IC6T = {k: ic_series(S, TGT[('ret', 6)]) for k, S in TZ.items()}
IC3T = {k: ic_series(S, TGT[('ret', 3)]) for k, S in TZ.items()}
wf, picks = [], []
for Y in range(2014, AS_OF.year + 1):
    tr = pd.Period(f'{Y-1}-06', 'M')
    cand = {k: v[v.index <= tr] for k, v in IC6T.items()}; cand = {k: v for k, v in cand.items() if len(v) >= 48}
    if not cand: continue
    best = max(cand, key=lambda k: cand[k].mean()); picks.append({'year': Y, 'selected': best, 'training IC': cand[best].mean()})
    for h, ICs in ((3, IC3T), (6, IC6T)):
        f = ICs[best]; f = f[(f.index >= pd.Period(f'{Y}-01', 'M')) & (f.index <= pd.Period(f'{Y}-12', 'M'))]
        for t, v in f.items(): wf.append({'horizon': f'{h}M', 'month': t, 'IC': v, 'signal used': best})
TPICKS = pd.DataFrame(picks); display(TPICKS.set_index('year'))
TWF = pd.DataFrame(wf)
g = TWF.groupby('horizon')['IC'].agg(['mean', 'count']); g['HAC t'] = [hac_t(TWF[TWF.horizon == h]['IC'].values, int(h[0])) for h in g.index]
display(g.round(3)); TPICKS.to_csv(OUT/'technical_wf_picks.csv', index=False); TWF.to_csv(OUT/'technical_wf_monthly.csv', index=False)
print(f'Distinct candidates selected across {len(TPICKS)} years: {TPICKS.selected.nunique()} -> ' + ('UNSTABLE' if TPICKS.selected.nunique() > 2 else 'stable'))

,selected,training IC
year,,
2014,B 12−1M momentum,0.115
2015,B 12−1M momentum,0.087
2016,B 12−1M momentum,0.128
2017,E Signed trend quality 12M,0.134
2018,E Signed trend quality 12M,0.109
2019,B 12−1M momentum,0.128
2020,B 12−1M momentum,0.137
2021,B 12−1M momentum,0.169
2022,B 12−1M momentum,0.157


,mean,count,HAC t
horizon,,,
3M,0.099,145,1.810
6M,0.113,142,1.850


Distinct candidates selected across 13 years: 2 -> stable


## 4. Reconciling the technical examined IC (+0.163) with the walk-forward IC (+0.058)
These are different calculations on different samples. Set out explicitly.

In [8]:
TQ = TZ['E Signed trend quality 12M']
ic6 = ic_series(TQ, TGT[('ret', 6)])
wf_months = pd.PeriodIndex(sorted(TWF[TWF.horizon == '6M']['month'].unique()))
exam = ic6[ic6.index >= EXAM_START]; onwf = ic6[ic6.index.isin(wf_months)]
REC = pd.DataFrame([
 {'figure': 'Examined-window IC (+0.163)', 'signal': 'Signed trend quality only — one fixed construction', 'sample': f'signal months {exam.index.min()}..{exam.index.max()}', 'months': len(exam), 'IC': exam.mean()},
 {'figure': 'Walk-forward IC (+0.058)', 'signal': f'whichever of the 6 candidates was selected that year ({TPICKS.selected.nunique()} different ones used)', 'sample': f'signal months {wf_months.min()}..{wf_months.max()}', 'months': len(onwf), 'IC': TWF[TWF.horizon == "6M"]["IC"].mean()},
 {'figure': 'LIKE-FOR-LIKE: trend quality on the walk-forward months', 'signal': 'Signed trend quality only', 'sample': f'signal months {wf_months.min()}..{wf_months.max()}', 'months': len(onwf), 'IC': onwf.mean()},
]).set_index('figure')
display(REC.round(3)); REC.to_csv(OUT/'technical_reconciliation.csv')
share = (TPICKS.selected == 'E Signed trend quality 12M').mean()
print(f'The gap is NOT a contradiction: the two numbers measure different things.')
print(f' - The +0.163 is one fixed construction over the examined window.')
print(f' - The +0.058 is the result of CHOOSING a technical construction each year; trend quality was the choice in only {share:.0%} of years.')
print(f' - On the identical walk-forward months, trend quality alone scores {onwf.mean():+.3f}.')
print('So trend quality itself is not the problem; knowing IN ADVANCE which technical construction to use is the problem.')
print('The selection instability is the honest cost, and it is why the walk-forward number is the one to quote.')

,signal,sample,months,IC
figure,,,,
Examined-window IC (+0.163),Signed trend quality only — one fixed construc...,signal months 2017-07..2025-10,100,0.163
Walk-forward IC (+0.058),whichever of the 6 candidates was selected tha...,signal months 2014-01..2025-10,142,0.113
LIKE-FOR-LIKE: trend quality on the walk-forward months,Signed trend quality only,signal months 2014-01..2025-10,142,0.175


The gap is NOT a contradiction: the two numbers measure different things.
 - The +0.163 is one fixed construction over the examined window.
 - The +0.058 is the result of CHOOSING a technical construction each year; trend quality was the choice in only 46% of years.
 - On the identical walk-forward months, trend quality alone scores +0.175.
So trend quality itself is not the problem; knowing IN ADVANCE which technical construction to use is the problem.
The selection instability is the honest cost, and it is why the walk-forward number is the one to quote.


### 4a. Why v15 reported +0.058 and this section reports +0.113
The two walk-forwards search different candidate sets. v15 searched 12 technical candidates (6 constructions × 2 standardisation routes, including low volatility and drawdown).
This section searches the 6 economically coherent price-path measures with directions frozen. A smaller, better-motivated menu is easier to choose from, so selection is more stable.

In [9]:
def trendq_pit(P, w=12): return pit_z(trend_q(P, w))
V15SET = {'T0 low vol CS': -RLOC.rolling(12).std(), 'T1 mom6 CS': PX / PX.shift(6) - 1, 'T2 mom12-1 CS': PX.shift(1) / PX.shift(13) - 1,
          'T3 voladj CS': (PX.shift(1) / PX.shift(13) - 1) / RLOC.rolling(12).std(), 'T4 trendq CS': trend_q(PX, 12),
          'T5 drawdown CS': PX.rolling(12).apply(lambda s: (s / np.maximum.accumulate(s) - 1).min(), raw=True)}
V15Z = {k: cs_z(v.reindex(IDX)) for k, v in V15SET.items()} | {k.replace(' CS', ' PIT'): cs_z(pit_z(v.reindex(IDX))) for k, v in V15SET.items()}
IC15 = {k: ic_series(S, TGT[('ret', 6)]) for k, S in V15Z.items()}
wf15, pk15 = [], []
for Y in range(2014, AS_OF.year + 1):
    tr = pd.Period(f'{Y-1}-06', 'M'); cand = {k: v[v.index <= tr] for k, v in IC15.items()}; cand = {k: v for k, v in cand.items() if len(v) >= 48}
    if not cand: continue
    best = max(cand, key=lambda k: cand[k].mean()); pk15.append(best)
    f = IC15[best]; wf15 += list(f[(f.index >= pd.Period(f'{Y}-01', 'M')) & (f.index <= pd.Period(f'{Y}-12', 'M'))].values)
REC2 = pd.DataFrame([
 {'walk-forward variant': 'v15: 12 candidates (6 constructions × 2 routes, incl. low vol and drawdown)', 'distinct picks': len(set(pk15)), 'months': len(wf15), 'IC 6M': np.mean(wf15)},
 {'walk-forward variant': 'v16: 6 price-path measures, directions frozen on discovery', 'distinct picks': TPICKS.selected.nunique(), 'months': len(TWF[TWF.horizon == '6M']), 'IC 6M': TWF[TWF.horizon == '6M']['IC'].mean()},
]).set_index('walk-forward variant')
display(REC2.round(3)); REC2.to_csv(OUT/'technical_wf_reconciliation.csv')
print(f'v15 picks used {len(set(pk15))} different candidates; v16 picks used {TPICKS.selected.nunique()}. Both numbers are correct for their own candidate set.')

,distinct picks,months,IC 6M
walk-forward variant,,,
"v15: 12 candidates (6 constructions × 2 routes, incl. low vol and drawdown)",5,142,0.058
"v16: 6 price-path measures, directions frozen on discovery",2,142,0.113


v15 picks used 5 different candidates; v16 picks used 2. Both numbers are correct for their own candidate set.


### 4b. Technical verdict

In [10]:
TV = pd.DataFrame([
 {'test': 'Raw IC (best candidate: signed trend quality, 6M)', 'result': f"+{TBAT.loc[('E Signed trend quality 12M', '6M'), 'IC full']:.3f} full, +{TBAT.loc[('E Signed trend quality 12M', '6M'), 'IC exam']:.3f} examined, HAC t {TBAT.loc[('E Signed trend quality 12M', '6M'), 'HAC t']:.2f}", 'reading': 'looks like the strongest technical result'},
 {'test': 'Orthogonalised against Micro + Valuation', 'result': f"IC falls to +{ORTH.loc['E Signed trend quality 12M', '6M residual IC full']:.3f} full and +{ORTH.loc['E Signed trend quality 12M', '6M residual IC exam']:.3f} examined ({ORTH.loc['E Signed trend quality 12M', '6M IC retained']:.0%} retained)", 'reading': 'most of the signal is the other two pillars restated'},
 {'test': 'Leave-one-country-out (US removed)', 'result': f"IC falls from +{TBAT.loc[('E Signed trend quality 12M', '6M'), 'IC full']:.3f} to +{TBAT.loc[('E Signed trend quality 12M', '6M'), 'ex-US']:.3f}", 'reading': 'almost entirely a US effect'},
 {'test': 'Every other candidate, ex-US', 'result': f"6M momentum {TBAT.loc[('A 6M momentum', '6M'), 'ex-US']:+.3f}, 12−1M {TBAT.loc[('B 12−1M momentum', '6M'), 'ex-US']:+.3f}, vol-adjusted {TBAT.loc[('F Vol-adjusted mom 12−1M', '6M'), 'ex-US']:+.3f}", 'reading': 'no momentum variant survives without the US'},
 {'test': 'Reversal hypotheses (1M, 3M)', 'result': f"full IC {TBAT.loc[('C 1M return', '6M'), 'IC full']:+.3f} and {TBAT.loc[('D 3M return', '6M'), 'IC full']:+.3f}", 'reading': 'reversal is not present at country level in this dataset'},
 {'test': 'Price-series independence', 'result': 'price = P/E × EPS exactly; the EPS leg carries 52% of 6M variance and the P/E leg 48%', 'reading': 'a technical pillar cannot be independent of Micro and Valuation here'},
]).set_index('test')
display(TV); TV.to_csv(OUT/'technical_verdict.csv')
print('CONCLUSION: no independent technical signal survives in the supplied data.')
print('The strongest candidate (signed trend quality) is retained for the scorecard as a DIAGNOSTIC indicator only, with no predictive weight.')
print('This is a limitation of the dataset — it contains no index price — not evidence that technical signals do not work in general.')

,result,reading
test,,
"Raw IC (best candidate: signed trend quality, 6M)","+0.133 full, +0.163 examined, HAC t 2.46",looks like the strongest technical result
Orthogonalised against Micro + Valuation,IC falls to +0.057 full and +0.039 examined (4...,most of the signal is the other two pillars re...
Leave-one-country-out (US removed),IC falls from +0.133 to +0.007,almost entirely a US effect
"Every other candidate, ex-US","6M momentum -0.045, 12−1M -0.013, vol-adjusted...",no momentum variant survives without the US
"Reversal hypotheses (1M, 3M)",full IC -0.014 and -0.013,reversal is not present at country level in th...
Price-series independence,price = P/E × EPS exactly; the EPS leg carries...,a technical pillar cannot be independent of Mi...


CONCLUSION: no independent technical signal survives in the supplied data.
The strongest candidate (signed trend quality) is retained for the scorecard as a DIAGNOSTIC indicator only, with no predictive weight.
This is a limitation of the dataset — it contains no index price — not evidence that technical signals do not work in general.


## 5. Micro — do acceleration and revision momentum add to the 6M revision?

In [11]:
R3 = EPS / EPS.shift(3) - 1; R6 = EPS / EPS.shift(6) - 1; R12 = EPS / EPS.shift(12) - 1
MICRO_C = {'Mi1 EPS revision 6M (incumbent)': R6, 'Mi2 EPS revision 3M': R3, 'Mi3 EPS revision 12M': R12,
           'Mi4 Revision acceleration (3M − prior 3M)': R3 - (EPS.shift(3) / EPS.shift(6) - 1),
           'Mi5 Revision momentum (3M pace − 6M pace)': R3 - R6 / 2,
           'Mi6 Revision breadth proxy (sign-weighted 6M)': np.sign(EPS.diff()).rolling(6).sum()}
rows = []
for k, raw in MICRO_C.items():
    for route, S in (('CS', cs_z(raw.reindex(IDX))), ('PIT→CS', cs_z(pit_z(raw.reindex(IDX))))):
        for h in (3, 6):
            s = summarise(S, h)
            loos = {c: (lambda keep: ic_series(cs_z((pit_z(raw[keep]) if route == 'PIT→CS' else raw[keep]).reindex(IDX)),
                                               (lambda y: y.sub(y.mean(axis=1), axis=0))(PX_USD[keep].shift(-h) / PX_USD[keep] - 1)).mean())([c2 for c2 in COUNTRIES if c2 != c]) for c in COUNTRIES}
            rows.append({'construction': k, 'route': route, 'horizon': f'{h}M', **{kk: s[kk] for kk in ('n', 'IC disc', 'IC exam', 'IC full', 'HAC t')},
                         'ex-US': loos['US'], 'min LOO': min(loos.values())})
MBAT = pd.DataFrame(rows).set_index(['construction', 'route', 'horizon'])
display(MBAT.style.format({c: '{:+.3f}' for c in MBAT.columns if c not in ('n', 'HAC t')} | {'HAC t': '{:+.2f}', 'n': '{:.0f}'}))
MBAT.to_csv(OUT/'micro_battery.csv')
INC_M = []
base = cs_z(R6)
for k, raw in MICRO_C.items():
    if k.startswith('Mi1'): continue
    add = cs_z(raw.reindex(IDX))
    for h in (3, 6):
        a = ic_series(base, TGT[('ret', h)]); b = ic_series(cs_z((base + add) / 2), TGT[('ret', h)])
        j = pd.concat([a.rename('a'), b.rename('b')], axis=1).dropna(); d = j.b - j.a
        INC_M.append({'added to Mi1': k, 'horizon': f'{h}M', 'IC Mi1 alone': j.a.mean(), 'IC blended': j.b.mean(),
                      'ΔIC': d.mean(), 'HAC t': hac_t(d.values, h), 'ΔIC discovery': d[d.index <= EMB_END].mean(), 'ΔIC examined': d[d.index >= EXAM_START].mean()})
INCM = pd.DataFrame(INC_M).set_index(['added to Mi1', 'horizon'])
display(INCM.style.format({c: '{:+.3f}' for c in INCM.columns if c != 'HAC t'} | {'HAC t': '{:+.2f}'}))
INCM.to_csv(OUT/'micro_incremental.csv')
print('A candidate earns a place only if ΔIC is positive in BOTH windows with a supporting t-statistic.')

A candidate earns a place only if ΔIC is positive in BOTH windows with a supporting t-statistic.


### 5a. Micro verdict
Acceleration and revision momentum are **refuted**: both have negative IC and both *subtract* from the incumbent when blended (ΔIC −0.09, t −2.4 at 3M).
A breadth proxy (count of monthly EPS increases over six months) was also tested; it has attractive raw numbers but its incremental ΔIC over the incumbent is not significant (t 0.95 / 0.55) and it is more US-dependent (ex-US +0.078 versus +0.105). **Not promoted.**
**Micro stays: EPS revision 6M, cross-sectional standardisation.**

## 6. Valuation — resolving the negative out-of-sample result
Spread $= \tfrac{1}{\text{forward P/E}} - \text{2Y government yield}$ (a valuation **proxy**, not an observed equity risk premium: no dividends, no payout data, consensus forward EPS).

In [12]:
EY = 100.0 / PE; SPREAD = EY - Y2
VAR = {'own-history z, 36M burn-in': cs_z(pit_z(SPREAD, 36)), 'own-history z, 60M burn-in': cs_z(pit_z(SPREAD, 60)),
       'own-history z, 120M burn-in': cs_z(pit_z(SPREAD, 120)), 'cross-sectional z only': cs_z(SPREAD),
       'cross-country RANK only': cs_z(SPREAD.rank(axis=1)), 'own-history z then rank': cs_z(pit_z(SPREAD, 36).rank(axis=1)),
       'winsorised own-history z (±2σ)': cs_z(pit_z(SPREAD, 36).clip(-2, 2)), 'earnings yield only (no 2Y)': cs_z(pit_z(EY, 36))}
rows = []
for k, S in VAR.items():
    for h in (3, 6):
        s = summarise(S, h); sm_ = summarise(S, h, 'mult')
        rows.append({'variant': k, 'horizon': f'{h}M', **{kk: s[kk] for kk in ('n', 'IC disc', 'IC exam', 'IC full', 'HAC t')},
                     'multiple-expansion IC disc': sm_['IC disc'], 'multiple-expansion IC exam': sm_['IC exam']})
VBAT = pd.DataFrame(rows).set_index(['variant', 'horizon'])
display(VBAT.style.format({c: '{:+.3f}' for c in VBAT.columns if c not in ('n', 'HAC t')} | {'HAC t': '{:+.2f}', 'n': '{:.0f}'}))
VBAT.to_csv(OUT/'valuation_variants.csv')
print('Every variant shows the same pattern: positive in discovery, negative in the examined window, for TOTAL RETURN.')
print('Against MULTIPLE EXPANSION the examined-window IC is far less negative — the spread does partly predict re-rating; the earnings leg cancels it.')

Every variant shows the same pattern: positive in discovery, negative in the examined window, for TOTAL RETURN.
Against MULTIPLE EXPANSION the examined-window IC is far less negative — the spread does partly predict re-rating; the earnings leg cancels it.


### 6a. Where does the negative examined IC come from?

In [13]:
BASE_V = VAR['own-history z, 36M burn-in']
parts6 = ic_parts(BASE_V, TGT[('ret', 6)])
CTRY = pd.DataFrame({'discovery contribution': parts6[parts6.index <= EMB_END].mean(), 'examined contribution': parts6[parts6.index >= EXAM_START].mean()})
CTRY.loc['TOTAL = IC'] = CTRY.sum(); display(CTRY.round(3))
sub = []
for lab, a, b in [('2010-2013', '2010-01', '2013-12'), ('2014-2016', '2014-01', '2016-12'), ('2017-2019', '2017-01', '2019-12'),
                  ('2020-2022', '2020-01', '2022-12'), ('2023-2026', '2023-01', '2026-04')]:
    ic = ic_series(BASE_V, TGT[('ret', 6)]); m = ic[(ic.index >= pd.Period(a, 'M')) & (ic.index <= pd.Period(b, 'M'))]
    negrate = (Y2 < 0).reindex(m.index).mean(axis=1).mean()
    sub.append({'period': lab, 'months': len(m), 'IC 6M': m.mean(), 'mean share of markets with NEGATIVE 2Y yield': negrate})
SUB = pd.DataFrame(sub).set_index('period'); display(SUB.round(3)); SUB.to_csv(OUT/'valuation_subperiods.csv')
print('HYPOTHESIS TESTED: the nominal 2Y leg misbehaves under negative policy rates (a large share of the panel, 2014-2022).')
j = pd.concat([ic_series(BASE_V, TGT[('ret', 6)]).rename('ic'), (Y2 < 0).mean(axis=1).rename('neg')], axis=1).dropna()
print(f'Correlation between the monthly IC and the share of markets with negative 2Y yields: {j.ic.corr(j.neg):+.2f}')
print(f'IC when NO market has a negative 2Y: {j[j.neg == 0].ic.mean():+.3f} ({(j.neg == 0).sum()} months) | when at least one does: {j[j.neg > 0].ic.mean():+.3f} ({(j.neg > 0).sum()} months)')
print('-> REFUTED. Negative rates are not the cause. The country decomposition above identifies the US as the source:')
print(f'   the US contributes {CTRY.loc["US", "examined contribution"]:+.3f} of the {CTRY.loc["TOTAL = IC", "examined contribution"]:+.3f} examined IC, while the other four are collectively positive.')

,discovery contribution,examined contribution
US,0.063,-0.078
Eurozone,0.057,0.011
UK,0.087,-0.003
Switzerland,0.005,-0.003
Japan,0.089,0.020
TOTAL = IC,0.301,-0.053


,months,IC 6M,mean share of markets with NEGATIVE 2Y yield
period,,,
2010-2013,48,0.219,0.292
2014-2016,36,0.314,0.594
2017-2019,36,-0.139,0.650
2020-2022,36,0.150,0.711
2023-2026,34,-0.094,0.076


HYPOTHESIS TESTED: the nominal 2Y leg misbehaves under negative policy rates (a large share of the panel, 2014-2022).
Correlation between the monthly IC and the share of markets with negative 2Y yields: -0.07
IC when NO market has a negative 2Y: +0.153 (34 months) | when at least one does: +0.126 (170 months)
-> REFUTED. Negative rates are not the cause. The country decomposition above identifies the US as the source:
   the US contributes -0.078 of the -0.053 examined IC, while the other four are collectively positive.


### 6b. Outlier sensitivity, leave-one-out and walk-forward

In [14]:
rows = []
for k in ('own-history z, 36M burn-in', 'winsorised own-history z (±2σ)', 'cross-country RANK only'):
    S = VAR[k]
    for h in (3, 6):
        ic = ic_series(S, TGT[('ret', h)]); srt = ic.sort_values()
        rows.append({'variant': k, 'horizon': f'{h}M', 'IC full': ic.mean(), 'IC drop worst 6 months': srt.iloc[6:].mean(),
                     'IC drop best 6 months': srt.iloc[:-6].mean(), 'IC median month': ic.median(), 'share months > 0': (ic > 0).mean()})
OUTL = pd.DataFrame(rows).set_index(['variant', 'horizon']); display(OUTL.round(3)); OUTL.to_csv(OUT/'valuation_outliers.csv')
rows = []
for h in (3, 6):
    r = {'horizon': f'{h}M', 'all 5': ic_series(BASE_V, TGT[('ret', h)]).mean()}
    for c in COUNTRIES:
        keep = [x for x in COUNTRIES if x != c]
        S = cs_z(pit_z(SPREAD[keep], 36)); y = PX_USD[keep].shift(-h) / PX_USD[keep] - 1; y = y.sub(y.mean(axis=1), axis=0)
        ic = ic_series(S, y); r[f'ex-{c}'] = ic.mean(); r[f'ex-{c} exam'] = ic[ic.index >= EXAM_START].mean()
    rows.append(r)
VLOO = pd.DataFrame(rows).set_index('horizon'); display(VLOO.round(3)); VLOO.to_csv(OUT/'valuation_loo.csv')
ICV = {k: ic_series(S, TGT[('ret', 6)]) for k, S in VAR.items()}
wfv, pkv = [], []
for Y in range(2014, AS_OF.year + 1):
    tr = pd.Period(f'{Y-1}-06', 'M'); cand = {k: v[v.index <= tr] for k, v in ICV.items()}; cand = {k: v for k, v in cand.items() if len(v) >= 48}
    if not cand: continue
    best = max(cand, key=lambda k: cand[k].mean()); pkv.append({'year': Y, 'selected': best})
    f = ICV[best]; wfv += list(f[(f.index >= pd.Period(f'{Y}-01', 'M')) & (f.index <= pd.Period(f'{Y}-12', 'M'))].values)
print(f'Valuation walk-forward (choose variant yearly from past data): IC 6M {np.mean(wfv):+.3f} over {len(wfv)} months, t {hac_t(np.array(wfv), 6):+.2f}')
print('Variants selected:', pd.Series([p['selected'] for p in pkv]).value_counts().to_dict())
VV = pd.DataFrame([
 {'question': 'Does own-history standardisation help?', 'answer': f"Yes. 6M full IC {VBAT.loc[('own-history z, 36M burn-in','6M'),'IC full']:+.3f} vs {VBAT.loc[('cross-sectional z only','6M'),'IC full']:+.3f} for cross-sectional only."},
 {'question': 'Does the 2Y leg help?', 'answer': f"Yes. Examined 6M IC {VBAT.loc[('own-history z, 36M burn-in','6M'),'IC exam']:+.3f} with the 2Y leg vs {VBAT.loc[('earnings yield only (no 2Y)','6M'),'IC exam']:+.3f} without it."},
 {'question': 'Is it driven by outliers?', 'answer': f"No. Median monthly IC {OUTL.loc[('own-history z, 36M burn-in','6M'),'IC median month']:+.3f}; dropping the best six months leaves {OUTL.loc[('own-history z, 36M burn-in','6M'),'IC drop best 6 months']:+.3f}."},
 {'question': 'Is it driven by one country?', 'answer': f"Full-sample: no — every leave-one-out IC stays positive (min {min(VLOO.loc['6M', [f'ex-{c}' for c in COUNTRIES]]):+.3f}). Out of sample: YES — the US alone contributes {CTRY.loc['US','examined contribution']:+.3f} of the {CTRY.loc['TOTAL = IC','examined contribution']:+.3f} examined IC, and EXCLUDING the US the examined 6M IC is {VLOO.loc['6M','ex-US exam']:+.3f}."},
 {'question': 'Is the negative out-of-sample result resolved?', 'answer': f"NO. The examined-window IC is {VBAT.loc[('own-history z, 36M burn-in','6M'),'IC exam']:+.3f} for total return and the walk-forward IC is {np.mean(wfv):+.3f}. The signal predicts MULTIPLE EXPANSION ({VBAT.loc[('own-history z, 36M burn-in','6M'),'multiple-expansion IC exam']:+.3f} examined) but the earnings leg offsets it."},
 {'question': 'What is the cause?', 'answer': f"The US. A negative-rate explanation was tested and REFUTED (IC {j[j.neg == 0].ic.mean():+.3f} when no market has a negative 2Y vs {j[j.neg > 0].ic.mean():+.3f} when some do). The US was persistently the most expensive market on this measure and persistently outperformed from 2017; the signal therefore ranked it last and was wrong. The other four markets behaved as the hypothesis predicts."},
 {'question': 'Does it work anywhere out of sample?', 'answer': f"Yes, outside the US: the ex-US examined 6M IC is {VLOO.loc['6M','ex-US exam']:+.3f}. That is a genuine but narrower claim, and it cannot be acted on in a five-market scorecard that must rank the US."},
]).set_index('question')
display(VV); VV.to_csv(OUT/'valuation_verdict.csv')

IC full  IC drop worst 6 months  IC drop best 6 months  IC median month  share months > 0
variant                        horizon                                                                                           
own-history z, 36M burn-in     3M         0.032                   0.060                  0.005            0.000             0.478
                               6M         0.131                   0.160                  0.106            0.200             0.574
winsorised own-history z (±2σ) 3M         0.031                   0.060                  0.005            0.000             0.478
                               6M         0.134                   0.163                  0.109            0.200             0.574
cross-country RANK only        3M        -0.004                   0.019                 -0.028            0.000             0.463
                               6M         0.039                   0.064                  0.015            0.000             0.494

,all 5,ex-US,ex-US exam,ex-Eurozone,ex-Eurozone exam,ex-UK,ex-UK exam,ex-Switzerland,ex-Switzerland exam,ex-Japan,ex-Japan exam
horizon,,,,,,,,,,,
3M,0.032,0.078,0.019,0.001,-0.126,-0.008,-0.113,0.063,-0.068,0.018,-0.163
6M,0.131,0.191,0.076,0.094,-0.084,0.101,-0.072,0.166,-0.028,0.092,-0.124


Valuation walk-forward (choose variant yearly from past data): IC 6M +0.060 over 142 months, t +0.78
Variants selected: {'winsorised own-history z (±2σ)': 13}


,answer
question,
Does own-history standardisation help?,Yes. 6M full IC +0.131 vs +0.039 for cross-sec...
Does the 2Y leg help?,Yes. Examined 6M IC -0.053 with the 2Y leg vs ...
Is it driven by outliers?,No. Median monthly IC +0.200; dropping the bes...
Is it driven by one country?,Full-sample: no — every leave-one-out IC stays...
Is the negative out-of-sample result resolved?,NO. The examined-window IC is -0.053 for total...
What is the cause?,The US. A negative-rate explanation was tested...
Does it work anywhere out of sample?,"Yes, outside the US: the ex-US examined 6M IC ..."


## 7. Frozen signal definitions for the final scorecard
Chosen on discovery and structural evidence, then frozen. The final notebook implements exactly these and nothing else.

In [15]:
FROZEN = pd.DataFrame([
 {'pillar': 'Macro', 'indicator': 'FCI 6M easing', 'formula': '(FCI_t − FCI_t−6) ÷ expanding sd of that change (≥36 obs), FCI oriented so higher = easier; then cross-sectional z',
  'direction': 'more easing ⇒ more attractive', 'evidence': 'IC 6M: discovery +0.186, examined +0.173, full +0.174 (HAC t 2.96, CI [+0.058, +0.299]); walk-forward +0.162 (t 2.26); every leave-one-out positive',
  'confidence label': 'PROVISIONAL — predictive', 'changed from v15?': 'no'},
 {'pillar': 'Micro', 'indicator': 'EPS revision 6M', 'formula': 'EPS_t / EPS_t−6 − 1 on 12M-forward consensus EPS; cross-sectional z',
  'direction': 'stronger upgrades ⇒ more attractive', 'evidence': 'IC 3M +0.131 (t 2.60), 6M +0.130 (t 1.92); walk-forward 3M +0.133, 6M +0.113; acceleration and revision momentum tested and REFUTED (ΔIC −0.09, t −2.4)',
  'confidence label': 'PROVISIONAL — predictive at 3M', 'changed from v15?': 'no'},
 {'pillar': 'Valuation', 'indicator': 'Earnings-yield spread vs own history', 'formula': '(100 / forward P/E − 2Y government yield), expanding within-country z (≥36 obs); then cross-sectional z',
  'direction': 'wider spread vs own history ⇒ more attractive', 'evidence': 'IC 6M: discovery +0.301, examined −0.053, full +0.131 (t 2.02); walk-forward +0.060 (t 0.78). Not outlier-driven. Negative out-of-sample result traced to the US; ex-US examined IC +0.076',
  'confidence label': 'DIAGNOSTIC — not predictive out of sample', 'changed from v15?': 'no (v15 construction retained)'},
 {'pillar': 'Technical', 'indicator': 'Signed trend quality 12M', 'formula': 'sign(slope) × R² of log reconstructed price on a 12-month time trend; cross-sectional z',
  'direction': 'smooth rising trend ⇒ more attractive (continuation; direction fixed on discovery)', 'evidence': 'Raw IC 6M +0.133 (t 2.46) but only +0.057 after removing Micro and Valuation, and +0.007 excluding the US. No momentum variant survives ex-US; reversal refuted',
  'confidence label': 'DIAGNOSTIC ONLY — no independent information', 'changed from v15?': 'no (same construction, now demonstrated to be non-independent)'},
]).set_index('pillar')
display(FROZEN); FROZEN.to_csv(OUT/'frozen_definitions.csv')
json.dump({'frozen': {r.name: {'indicator': r['indicator'], 'confidence': r['confidence label']} for _, r in FROZEN.iterrows()},
           'predictive_pillars': ['Macro', 'Micro'], 'diagnostic_only_pillars': ['Valuation', 'Technical'],
           'as_of': str(AS_OF), 'panel': [str(IDX[0]), str(AS_OF)], 'examined_window_is_not_holdout': True},
          open(OUT/'frozen_spec.json', 'w'), indent=2)
print('PREDICTIVE pillars (carry weight in the predictive view): Macro, Micro')
print('DIAGNOSTIC-ONLY pillars (shown, labelled, zero predictive weight): Valuation, Technical')
print('\nThe four-pillar equal-weight scorecard remains the primary CRISIL deliverable; the predictive view is reported alongside it.')

,indicator,formula,direction,evidence,confidence label,changed from v15?
pillar,,,,,,
Macro,FCI 6M easing,(FCI_t − FCI_t−6) ÷ expanding sd of that chang...,more easing ⇒ more attractive,"IC 6M: discovery +0.186, examined +0.173, full...",PROVISIONAL — predictive,no
Micro,EPS revision 6M,EPS_t / EPS_t−6 − 1 on 12M-forward consensus E...,stronger upgrades ⇒ more attractive,"IC 3M +0.131 (t 2.60), 6M +0.130 (t 1.92); wal...",PROVISIONAL — predictive at 3M,no
Valuation,Earnings-yield spread vs own history,"(100 / forward P/E − 2Y government yield), exp...",wider spread vs own history ⇒ more attractive,"IC 6M: discovery +0.301, examined −0.053, full...",DIAGNOSTIC — not predictive out of sample,no (v15 construction retained)
Technical,Signed trend quality 12M,sign(slope) × R² of log reconstructed price on...,smooth rising trend ⇒ more attractive (continu...,Raw IC 6M +0.133 (t 2.46) but only +0.057 afte...,DIAGNOSTIC ONLY — no independent information,"no (same construction, now demonstrated to be ..."


PREDICTIVE pillars (carry weight in the predictive view): Macro, Micro
DIAGNOSTIC-ONLY pillars (shown, labelled, zero predictive weight): Valuation, Technical

The four-pillar equal-weight scorecard remains the primary CRISIL deliverable; the predictive view is reported alongside it.
